## 4 — Gráficos: Perfil dos Estudantes


| Código | Tipo | Descrição |
|--------|------|-----------|
| 15 | Pirâmide etária | Faixa etária × sexo |
| 16 | Barras horizontais | Distribuição por cor/raça |
| 17 | Barras horizontais | Distribuição por renda familiar |
| 18 | Pizza (donut) | Distribuição por turno |
| 19 | Heatmap | Evasão — Faixa Etária × Renda Familiar |
| 20 | Heatmap | Evasão — Sexo × Turno |
| 21 | Heatmap | Evasão — Cor/Raça × Renda Familiar |
| 22 | Heatmap | Retenção — Faixa Etária × Renda Familiar |
| 23 | Heatmap | Retenção — Sexo × Turno |
| 24 | Heatmap | Retenção — Cor/Raça × Renda Familiar |
| 25 | Heatmap | Situação (Evasão/Retenção/Conclusão) × Perfil Sociodemográfico |

#### 4.1 Importações, configurações e paletas

In [5]:

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import sys
import os

sys.path.append(os.path.abspath(".."))

from utils import (
    carregar_dados,
    calcular_indicadores,
    gerar_mapa_cores,
    aplicar_layout_light,
    escala_indicador,
    CORES_INDICADORES,
    CORES_CATEGORIA,
    CORES_CONCLUSAO_LIGHT,
    CORES_FLUXO_LIGHT,
    CORES_MATRICULAS_ATIVAS_LIGHT,
    CORES_SITUACAO_LIGHT,
    COR_TEMPO_MEDIANO,
    CORES_CURSO,
    CORES_SITUACAO,
    PALETA_QUALITATIVA_LIGHT,
    ROTULOS_INDICADORES,
)

ORDEM_ETARIA = [
    "Menor de 14 anos", "15 a 19 anos", "20 a 24 anos", "25 a 29 anos", "30 a 34 anos",
    "35 a 39 anos", "40 a 44 anos", "45 a 49 anos", "50 a 54 anos", "55 a 59 anos", "Maior de 60 anos",
]
ORDEM_RENDA = ["0<RFP<=0,5", "0,5<RFP<=1", "1<RFP<=1,5", "1,5<RFP<=2,5", "2,5<RFP<=3,5", "RFP>3,5", "Não declarada"]
CORES_SEXO = {"F": "#C62828", "M": "#2E7D32"}

def heatmap_cruzado(df, col_y, col_x, situacao, ordem_y=None, ordem_x=None, n_minimo=5):
    if situacao == "Evadidos":
        mask = df["Categoria da Situação"] == "Evadidos"
    elif situacao == "Retido":
        mask = df["Situação de Matrícula"] == "Retido"
    else:
        raise ValueError("situação deve ser 'Evadidos' ou 'Retido'")

    contagem = (
        df[mask].groupby([col_y, col_x])["Código da Matricula"].nunique().reset_index(name="N")
    )
    total = df.groupby([col_y, col_x])["Código da Matricula"].nunique().reset_index(name="Total")
    merged = total.merge(contagem, on=[col_y, col_x], how="left").fillna({"N": 0})
    merged["Taxa (%)"] = np.where(merged["Total"] >= n_minimo, merged["N"] / merged["Total"] * 100, np.nan)

    pivot = merged.pivot_table(index=col_y, columns=col_x, values="Taxa (%)", aggfunc="mean")
    if ordem_y:
        pivot = pivot.reindex([x for x in ordem_y if x in pivot.index])
    if ordem_x:
        pivot = pivot[[x for x in ordem_x if x in pivot.columns]]
    return pivot.round(1)

def fig_heatmap(pivot, titulo, escala):
    fig = px.imshow(pivot, text_auto='.1f', aspect='auto', color_continuous_scale=escala, labels=dict(color='Taxa (%)'))
    fig.update_layout(title=titulo)
    aplicar_layout_light(fig, altura=520)
    return fig


pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

2026-05-25 10:48:45.018 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


#### 4.2 Carga dos dados tratados

In [6]:
df_completo = pd.read_parquet("../dados_tratados/restinga_dados_tratados.parquet")
print("Shape:", df_completo.shape)
df_completo.head(3)

Shape: (10285, 18)


,Ano,Categoria da Situação,Cor / Raça,Código da Matricula,Código do Ciclo Matricula,Data de Fim Previsto do Ciclo,Data de Inicio do Ciclo,Data de Ocorrencia da Matricula,Eixo Tecnológico,Faixa Etária,Mês De Ocorrência da Situação,Nome de Curso,Renda Familiar,Sexo,Situação de Matrícula,Subeixo Tecnológico,Tipo de Curso,Turno
0,2017,Em curso,Não declarada,44681986,1207788,2014-12-22,2012-03-05,2012-03-01,Informação e Comunicação,35 a 39 anos,2018-03-05,Análise e Desenvolvimento de Sistemas,Não declarada,F,Retido,Informática,Superior-Tecnologia,Matutino
1,2017,Em curso,Branca,66262145,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino
2,2017,Em curso,Branca,66262155,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino


#### 4.3 Filtros utilizados

In [7]:
ano_selecionado = int(df_completo['Ano'].max())
tipos_selecionados = sorted(df_completo['Tipo de Curso'].dropna().unique())
cursos_selecionados = sorted(df_completo['Nome de Curso'].dropna().unique())
n_minimo = 5

df = df_completo[
    (df_completo['Ano'] == ano_selecionado)
    & (df_completo['Tipo de Curso'].isin(tipos_selecionados))
    & (df_completo['Nome de Curso'].isin(cursos_selecionados))
].copy()

print('Ano selecionado:', ano_selecionado)
print('Matrículas distintas:', df['Código da Matricula'].nunique())
print('Cursos:', df['Nome de Curso'].nunique())

Ano selecionado: 2024
Matrículas distintas: 1745
Cursos: 11


#### 4.4 Gráficos

#### Gráfico 15 — Pirâmide etária por sexo

In [8]:
faixas_presentes = [f for f in ORDEM_ETARIA if f in df['Faixa Etária'].dropna().unique()]
piramide_df = df.groupby(['Faixa Etária', 'Sexo'])['Código da Matricula'].nunique().reset_index(name='Qtd')
piramide_df['Faixa Etária'] = pd.Categorical(piramide_df['Faixa Etária'], categories=faixas_presentes, ordered=True)
piramide_df = piramide_df.sort_values('Faixa Etária')
fig_g15 = go.Figure()
sexos = sorted(piramide_df['Sexo'].dropna().unique())
for sexo in sexos:
    subset = piramide_df[piramide_df['Sexo'] == sexo]
    multiplicador = -1 if sexo == sexos[0] else 1
    fig_g15.add_trace(go.Bar(name=sexo, y=subset['Faixa Etária'].astype(str), x=subset['Qtd'] * multiplicador, orientation='h', marker_color=CORES_SEXO.get(sexo, '#757575'), text=subset['Qtd'].astype(str), textposition='inside'))
valor_max = int(piramide_df['Qtd'].max()) if len(piramide_df) else 10
ticks = list(range(-valor_max, valor_max + max(1, valor_max // 5), max(1, valor_max // 5)))
fig_g15.update_layout(barmode='relative', xaxis=dict(tickvals=ticks, ticktext=[str(abs(v)) for v in ticks], title='Matrículas'), yaxis=dict(title=''))
aplicar_layout_light(fig_g15, altura=430)
fig_g15.show()

#### Gráficos 16 a 18 — Distribuições gerais

In [14]:
raca_df = df["Cor / Raça"].value_counts().reset_index()
raca_df.columns = ["Cor/Raça", "Qtd"]
raca_df["Percentual"] = raca_df["Qtd"] / raca_df["Qtd"].sum() * 100
raca_df["Texto"] = raca_df.apply(
    lambda linha: f"{int(linha['Qtd'])} ({linha['Percentual']:.1f}%)",
    axis=1,
)

fig_g16 = px.bar(
    raca_df.sort_values("Qtd"),
    x="Qtd",
    y="Cor/Raça",
    orientation="h",
    color="Qtd",
    color_continuous_scale="YlGnBu",
    text="Texto",
    labels={"Qtd": "Matrículas"},
)
fig_g16.update_traces(textposition="outside", cliponaxis=False)
fig_g16.update_xaxes(range=[0, raca_df["Qtd"].max() * 1.25])
fig_g16.update_layout(coloraxis_showscale=False)
aplicar_layout_light(fig_g16, altura=430)
fig_g16.show()


renda_df = (
    df["Renda Familiar"]
    .value_counts()
    .reindex([r for r in ORDEM_RENDA if r in df["Renda Familiar"].unique()])
    .reset_index()
)
renda_df.columns = ["Renda", "Qtd"]
renda_df["Percentual"] = renda_df["Qtd"] / renda_df["Qtd"].sum() * 100
renda_df["Texto"] = renda_df.apply(
    lambda linha: f"{int(linha['Qtd'])} ({linha['Percentual']:.1f}%)",
    axis=1,
)

fig_g17 = px.bar(
    renda_df.sort_values("Qtd"),
    x="Qtd",
    y="Renda",
    orientation="h",
    color="Qtd",
    color_continuous_scale="YlGnBu",
    text="Texto",
    labels={"Qtd": "Matrículas"},
)
fig_g17.update_traces(textposition="outside", cliponaxis=False)
fig_g17.update_xaxes(range=[0, renda_df["Qtd"].max() * 1.25])
fig_g17.update_layout(coloraxis_showscale=False)
aplicar_layout_light(fig_g17, altura=430)
fig_g17.show()


turno_df = df["Turno"].value_counts().reset_index()
turno_df.columns = ["Turno", "Qtd"]
turno_df["Percentual"] = turno_df["Qtd"] / turno_df["Qtd"].sum() * 100
turno_df["Texto"] = turno_df.apply(
    lambda linha: f"{int(linha['Qtd'])} ({linha['Percentual']:.1f}%)",
    axis=1,
)

fig_g18 = px.bar(
    turno_df,
    x="Turno",
    y="Qtd",
    color="Turno",
    color_discrete_map=gerar_mapa_cores(turno_df["Turno"]),
    text="Texto",
    labels={"Qtd": "Matrículas"},
)
fig_g18.update_traces(textposition="outside", cliponaxis=False)
fig_g18.update_layout(coloraxis_showscale=False)
aplicar_layout_light(fig_g18, altura=430)
fig_g18.show()

#### Gráficos 19 a 21 — Heatmaps de evasão

Células com menos de `n_minimo` matrículas são deixadas em branco para evitar interpretações instáveis.

In [10]:
p19 = heatmap_cruzado(df, 'Faixa Etária', 'Renda Familiar', 'Evadidos', ORDEM_ETARIA, ORDEM_RENDA, n_minimo)
fig_heatmap(p19, 'Taxa de Evasão — Faixa Etária × Renda Familiar', 'OrRd').show()

p20 = heatmap_cruzado(df, 'Sexo', 'Turno', 'Evadidos', None, None, n_minimo)
fig_heatmap(p20, 'Taxa de Evasão — Sexo × Turno', 'OrRd').show()

p21 = heatmap_cruzado(df, 'Cor / Raça', 'Renda Familiar', 'Evadidos', None, ORDEM_RENDA, n_minimo)
fig_heatmap(p21, 'Taxa de Evasão — Cor/Raça × Renda Familiar', 'OrRd').show()

#### Gráficos 22 a 24 — Heatmaps de retenção

In [11]:
p22 = heatmap_cruzado(df, 'Faixa Etária', 'Renda Familiar', 'Retido', ORDEM_ETARIA, ORDEM_RENDA, n_minimo)
fig_heatmap(p22, 'Taxa de Retenção — Faixa Etária × Renda Familiar', 'YlOrBr').show()

p23 = heatmap_cruzado(df, 'Sexo', 'Turno', 'Retido', None, None, n_minimo)
fig_heatmap(p23, 'Taxa de Retenção — Sexo × Turno', 'YlOrBr').show()

p24 = heatmap_cruzado(df, 'Cor / Raça', 'Renda Familiar', 'Retido', None, ORDEM_RENDA, n_minimo)
fig_heatmap(p24, 'Taxa de Retenção — Cor/Raça × Renda Familiar', 'YlOrBr').show()

#### Gráfico 25 — Matriz consolidada Situação × Perfil

Esta matriz mostra, para cada categoria de perfil, a composição percentual das situações de matrícula.

In [12]:
variaveis = ['Sexo', 'Cor / Raça', 'Faixa Etária', 'Renda Familiar', 'Turno']
linhas = []
for variavel in variaveis:
    total_por_cat = df.groupby(variavel)['Código da Matricula'].nunique()
    for categoria, total in total_por_cat.items():
        if total < n_minimo:
            continue
        for situacao in ['Em curso', 'Concluintes', 'Evadidos']:
            n = df[(df[variavel] == categoria) & (df['Categoria da Situação'] == situacao)]['Código da Matricula'].nunique()
            linhas.append({'Variável': variavel, 'Categoria': categoria, 'Situação': situacao, 'Taxa (%)': n / total * 100})
matriz = pd.DataFrame(linhas)
matriz['Perfil'] = matriz['Variável'] + ' — ' + matriz['Categoria'].astype(str)
pivot20 = matriz.pivot_table(index='Perfil', columns='Situação', values='Taxa (%)', aggfunc='mean').round(1)
fig_25 = px.imshow(pivot20, text_auto='.1f', aspect='auto', color_continuous_scale='Purples', labels=dict(color='Percentual (%)'))
aplicar_layout_light(fig_25, altura=720)
fig_25.show()